# 01 - Data Collection

### Objective

Inspect and validate cached NBA datasets used to build the player archetype profiles.

The project uses NBA base, advanced, tracking, shot-location, hustle, and matchup data to capture playing style behavior.

### Goals

1. Document the data sources.
2. Define the eligible player population.
3. Validate dataset structure.
4. Evaluate missing data and player coverage.
5. Identify data limitations before feature engineering.

No modeling or final feature construction is performed here.

### Data Collection

NBA API requests, validation, and caching are implemented in src/nba_archetypes/data_collection.py and executed through scripts/collect_data.py. 

The scripts/prepare_data.py script creates the eligible-player dataset and data-quality reports. 

This notebook reads those outputs, evaluates source coverage, and documents limitations before feature engineering.

### Inputs

- Cached NBA API datasets

### Outputs

(Prepared outputs reviewed within this notebook.)

- Eligible players dataset
- Dataset inventory
- Missing data report
- Column inventory
- Source manifest

In [1]:
import pandas as pd

from nba_archetypes.config import RAW_DATA_DIR, INTERIM_DATA_DIR, METADATA_DIR, MIN_MINUTES

import nba_archetypes.data_validation as dv

### 1. Data Sources and Collection

Player statistics were collected using the Python 'nba_api' package.

The collection pipeline retrieves several categories of data:
- Traditional and advanced player statistics
- Usage data and scoring splits
- Offensive tracking data
- Shot-location data
- Hustle and defensive tracking data
- Player matchups data

API requests and caching are managed through 'scripts/collect_data.py'.

The collection pipeline downloads and caches the 17 selected NBA datasets directly in Parquet format.

Collection settings and timestamps are recorded in source_manifest.csv, allowing the raw sources to be traced back to their collection configuration.

In [2]:
datasets = {}
for file in RAW_DATA_DIR.glob("*.parquet"):
    dataset_name = file.stem
    datasets[dataset_name] = pd.read_parquet(file)

player_types = ["base", "advanced", "scoring", "usage"]

tracking_types = [
    "passing", "rebounding", "speeddistance",
    "possessions", "drives", "catchshoot",
    "pullupshot", "posttouch", "painttouch",
    "efficiency"
]

expected_files = {f"{name}_stats.parquet" for name in player_types + tracking_types}
expected_files.update(["shot_locations.parquet", "hustle_stats.parquet", "matchup_stats.parquet"])

existing_files = {file.name for file in RAW_DATA_DIR.glob("*.parquet")}
missing_files = sorted(expected_files - existing_files)

print("Expected files:", len(expected_files))
print("Files found:", len(existing_files))
print("Missing expected files:", missing_files)

Expected files: 17
Files found: 17
Missing expected files: []


### 2. Data Validation

Player-level datasets are checked for required identifier columns and unique player IDs. These checks help establish whether the datasets can be joined without unintentionally duplicating player records.

The matchup dataset contains multiple observations per player and therefore does not require a unique PLAYER_ID. Its required columns are validated separately, and its observations will be aggregated before player-level integration.

In [3]:
data_val_report = []
for dataset_name, df in datasets.items():
    if dataset_name == "matchup_stats":
        unique_id = "Not applicable"
        required_cols = dv.check_required_cols(df, ["OFF_PLAYER_ID", "DEF_PLAYER_ID", "PARTIAL_POSS"])
    else:
        unique_id = dv.check_unique_ids(df, "PLAYER_ID")
        required_cols = dv.check_required_cols(df, ["PLAYER_ID", "PLAYER_NAME"])
    
    data_val_report.append({
        "dataset": dataset_name,
        "unique_player_id": unique_id,
        "required_columns": required_cols
    })

data_val_report = pd.DataFrame(data_val_report)
display(data_val_report)

,dataset,unique_player_id,required_columns
0,advanced_stats,True,True
1,base_stats,True,True
2,catchshoot_stats,True,True
3,drives_stats,True,True
4,efficiency_stats,True,True
5,hustle_stats,True,True
6,matchup_stats,Not applicable,True
7,painttouch_stats,True,True
8,passing_stats,True,True
9,possessions_stats,True,True


### 3. Dataset Inventory

Before cleaning or merging the datasets, let's examine the available source tables.

The inventory records the number of rows and columns in each dataset and helps identify missing or incomplete sources.

In [4]:
inventory = dv.dataset_inventory(datasets)

display(inventory)

,dataset,rows,columns,missing_values
0,advanced_stats,582,79,0
1,base_stats,582,69,0
2,catchshoot_stats,582,16,140
3,drives_stats,582,25,0
4,efficiency_stats,582,22,0
5,hustle_stats,581,28,0
6,matchup_stats,147805,27,0
7,painttouch_stats,582,26,0
8,passing_stats,582,18,0
9,possessions_stats,582,21,0


### 4. Missing Value Analysis

The dataset inventory report above reveals a few datasets containing missing values. We will investigate these a bit further in a later notebook but for now, let's document which columns contain missing values and the missing percentage for each.

Missing values at this stage aren't too alarming yet because they could be from columns that we will not use for our analysis at all or due to low activity which may be eliminated with our eligibility threshold.

We won't make any assumptions until we narrow down our player universe and feature set.

In [5]:
mv_report = dv.missing_value_report(datasets)

display(mv_report)

,dataset,column,missing_percentage
0,catchshoot_stats,CATCH_SHOOT_FG3_PCT,6.19
1,catchshoot_stats,CATCH_SHOOT_FG3M,6.01
2,catchshoot_stats,CATCH_SHOOT_FG3A,6.01
3,catchshoot_stats,CATCH_SHOOT_FG_PCT,2.92
4,catchshoot_stats,CATCH_SHOOT_EFG_PCT,2.92
5,pullupshot_stats,PULL_UP_FG3_PCT,17.18
6,pullupshot_stats,PULL_UP_FG_PCT,7.90
7,pullupshot_stats,PULL_UP_EFG_PCT,7.90
8,pullupshot_stats,PULL_UP_FG3M,6.01
9,pullupshot_stats,PULL_UP_FG3A,6.01


### 5. Player Universe

#### Eligibility Threshold

This project includes players with at least 250 regular season minutes.

This threshold is intended to reduce the influence of very small sample size players while retaining a reasonably broad pool of players to work with.

The threshold will be evaluated for sensitivity later in the modeling process.

#### Core Player Data

The eligible-player dataset is generated by scripts/prepare_data.py using the DuckDB query in sql/01_eligible_players.sql.

Our base and advanced stats tables will be used to establish the core player population and provide context such as minutes, games played, and possessions.

These datasets will also provide the identifiers needed to combine our other data sources.

In [6]:
print("Minutes threshold:", MIN_MINUTES)

eligible_players = pd.read_parquet(INTERIM_DATA_DIR / "eligible_players.parquet")

print("Total Players:", len(datasets["base_stats"]))
print("Eligible Players:", len(eligible_players))
print("Unique player IDs:", eligible_players["PLAYER_ID"].is_unique)
print("Missing possessions:", eligible_players["POSS"].isna().sum())
print("Non-positive possessions:", (eligible_players["POSS"] <= 0).sum())

display(eligible_players.head())

Minutes threshold: 250
Total Players: 582
Eligible Players: 450
Unique player IDs: True
Missing possessions: 0
Non-positive possessions: 0


,PLAYER_ID,PLAYER_NAME,TEAM_ABBREVIATION,AGE,GP,MIN,POSS
0,1631260,AJ Green,MIL,26.0,78,2269.933333,4704
1,1642358,AJ Johnson,DAL,21.0,48,453.433333,968
2,203932,Aaron Gordon,DEN,30.0,36,1004.958333,2096
3,1628988,Aaron Holiday,HOU,29.0,57,780.731667,1618
4,1630174,Aaron Nesmith,IND,26.0,45,1334.950000,2871


### 6. Shot Location Source Verification

The shot-location endpoint returns a different header structure from the other NBA datasets. The collection pipeline converts these hierarchical headers into standardized shooting-zone columns.

This section checks player coverage, uniqueness, and expected columns.

In [ ]:
shot_locations = datasets["shot_locations"]
shot_columns = [
    "RA_FGA",
    "PAINT_NON_RA_FGA",
    "MIDRANGE_FGA",
    "LEFT_CORNER_3_FGA",
    "RIGHT_CORNER_3_FGA",
    "ABOVE_BREAK_3_FGA",
    "BACKCOURT_FGA",
    "CORNER_3_FGA"
]

missing_columns = [
    col for col in shot_columns
    if col not in shot_locations.columns
]

eligible_ids = set(eligible_players["PLAYER_ID"])
shot_location_ids = set(shot_locations["PLAYER_ID"])

print("Shot-location rows:", len(shot_locations))
print("Duplicate player IDs:", shot_locations["PLAYER_ID"].duplicated().sum())
print("Missing shot-location columns:", missing_columns)
print("Eligible players missing:", len(eligible_ids - shot_location_ids))

Shot-location rows: 582
Duplicate player IDs: 0
Missing shot-location columns: []
Eligible players missing: 0
All shot-location columns: ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'NICKNAME', 'RA_FGM', 'RA_FGA', 'RA_FG_PCT', 'PAINT_NON_RA_FGM', 'PAINT_NON_RA_FGA', 'PAINT_NON_RA_FG_PCT', 'MIDRANGE_FGM', 'MIDRANGE_FGA', 'MIDRANGE_FG_PCT', 'LEFT_CORNER_3_FGM', 'LEFT_CORNER_3_FGA', 'LEFT_CORNER_3_FG_PCT', 'RIGHT_CORNER_3_FGM', 'RIGHT_CORNER_3_FGA', 'RIGHT_CORNER_3_FG_PCT', 'ABOVE_BREAK_3_FGM', 'ABOVE_BREAK_3_FGA', 'ABOVE_BREAK_3_FG_PCT', 'BACKCOURT_FGM', 'BACKCOURT_FGA', 'BACKCOURT_FG_PCT', 'CORNER_3_FGM', 'CORNER_3_FGA', 'CORNER_3_FG_PCT']


### 7. Player Data Coverage

Not every NBA dataset is guaranteed to include every eligible player. I measured the percentage of eligible players represented in each data source before deciding whether those variables should be included in our final model.

Player coverage measures whether an eligible player appears in a dataset, not whether every statistical field is populated. Missing values are evaluated separately.

The matchup_stats dataset has intentionally been left out of this report because an offensive player can matchup up against multiple defenders and a defender against multiple offensive players in that dataset.

In [ ]:
unique_player_datasets = {
    name: df 
    for name, df in datasets.items() 
    if name != "matchup_stats"
    }

display(dv.player_coverage_report(unique_player_datasets, eligible_players))

,dataset,coverage_percentage
0,advanced_stats,100.0
1,base_stats,100.0
2,catchshoot_stats,100.0
3,drives_stats,100.0
4,efficiency_stats,100.0
5,hustle_stats,100.0
6,painttouch_stats,100.0
7,passing_stats,100.0
8,possessions_stats,100.0
9,posttouch_stats,100.0


### 8. Matchup Data

As mentioned earlier, our matchup data is on a different granularity than the rest of our datasets. Unlike the player-level datasets, matchup records contain multiple rows per offensive/defensive player because they can matchup against many different opponents throughout the course of the season.

This section compares the sum of recorded defensive matchup possessions against each eligible player's total possessions. The resulting ratio provides an initial diagnostic of matchup-data coverage, although differences in the underlying possession definitions may affect its interpretation.

Matchup data will eventually be used to support engineered metrics such as defensive versatility, defensive matchup difficulty, and potentially more.

In [9]:
eligible_data = eligible_players.set_index("PLAYER_ID")
defender_poss = datasets["matchup_stats"].groupby("DEF_PLAYER_ID")["PARTIAL_POSS"].sum()

matchup_coverage = (defender_poss / eligible_data["POSS"]).describe()
display(matchup_coverage)

count    450.000000
mean       0.925925
std        0.019144
min        0.843886
25%        0.915140
50%        0.926677
75%        0.939496
max        0.964807
dtype: float64

### 9. Source Manifest

The source manifest documents the 17 raw datasets, including their NBA API endpoints, season settings, statistical modes, collection timestamps, and file sizes.

These records are generated during collection rather than inferred from the current configuration afterward. They provide an auditable record of the requested collection settings and help establish data provenance.

In [10]:
source_manifest = pd.read_csv(METADATA_DIR / "source_manifest.csv")

print("Collected sources:", len(source_manifest))

print("Missing provenance records:", sorted(expected_files - set(source_manifest["file"])))

display(source_manifest)

Collected sources: 17
Missing provenance records: []


,file,source,endpoint,season,season_type,per_mode,measure_type,distance_range,collected_at_utc,file_size_bytes
0,base_stats.parquet,NBA Stats API via nba_api,LeagueDashPlayerStats,2025-26,Regular Season,Totals,Base,NaN,2026-10-19T23:24:01+00:00,160299
1,advanced_stats.parquet,NBA Stats API via nba_api,LeagueDashPlayerStats,2025-26,Regular Season,Totals,Advanced,NaN,2026-10-19T23:24:03+00:00,221265
2,scoring_stats.parquet,NBA Stats API via nba_api,LeagueDashPlayerStats,2025-26,Regular Season,Totals,Scoring,NaN,2026-10-19T23:24:04+00:00,144756
3,usage_stats.parquet,NBA Stats API via nba_api,LeagueDashPlayerStats,2025-26,Regular Season,Totals,Usage,NaN,2026-10-19T23:24:08+00:00,148061
4,passing_stats.parquet,NBA Stats API via nba_api,LeagueDashPtStats,2025-26,Regular Season,Totals,Passing,NaN,2026-10-19T23:24:10+00:00,45687
5,rebounding_stats.parquet,NBA Stats API via nba_api,LeagueDashPtStats,2025-26,Regular Season,Totals,Rebounding,NaN,2026-10-19T23:24:14+00:00,80509
6,speeddistance_stats.parquet,NBA Stats API via nba_api,LeagueDashPtStats,2025-26,Regular Season,Totals,SpeedDistance,NaN,2026-10-19T23:24:16+00:00,43569
7,possessions_stats.parquet,NBA Stats API via nba_api,LeagueDashPtStats,2025-26,Regular Season,Totals,Possessions,NaN,2026-10-19T23:24:18+00:00,57088
8,drives_stats.parquet,NBA Stats API via nba_api,LeagueDashPtStats,2025-26,Regular Season,Totals,Drives,NaN,2026-10-19T23:24:19+00:00,57768
9,catchshoot_stats.parquet,NBA Stats API via nba_api,LeagueDashPtStats,2025-26,Regular Season,Totals,CatchShoot,NaN,2026-10-19T23:24:21+00:00,40501


### 10. Conclusion

#### Key Findings

- **Data Collection and Provenance:** All 17 selected NBA datasets were successfully collected for the 2025–26 regular season. Collection settings, API endpoints, timestamps, and file information are documented in the source manifest.

- **Data Validation:** The player-level datasets satisfy the required identifier and uniqueness checks. Matchup data have a different structure, containing multiple observations per player, and will require aggregation before integration with player-level statistics.

- **Missing Value Analysis:** Missing values are concentrated in the catch-and-shoot, pull-up shooting, and shot-location datasets. The highest observed missingness is approximately 17.2% for PULL_UP_FG3_PCT. The treatment of missing values will be determined after selecting and evaluating candidate features.

- **Player Universe:** Of the 582 players represented in the base dataset, 450 meet the primary eligibility threshold of 250 regular-season minutes. The resulting player table contains unique player identifiers and valid positive possession counts.

- **Player Coverage:** All 450 eligible players are represented in each of the 16 player-level datasets. This indicates complete player-ID coverage, although some individual statistical fields remain incomplete.

- **Shot-Location Data:** The refreshed dataset contains the expected shooting-zone columns and includes every eligible player. Additional checks of shooting-zone totals help assess internal consistency before engineering shot-diet features.

- **Matchup Data:** Recorded defensive matchup possessions average approximately 92.6% of the corresponding player possession totals. This provides an initial indication of source coverage, but further evaluation of measurement definitions and reliability will be necessary before constructing defensive features.

#### Decisions and Next Steps

- The 17 selected datasets will be retained as candidate sources for feature engineering.

- The next phase will focus on defining interpretable playing-style metrics, evaluating appropriate denominators, addressing missing values, and examining feature reliability and redundancy.

- Matchup observations will be aggregated to the player level before integration. The 250-minute eligibility threshold will also be evaluated through sensitivity analysis to determine whether lower-minute players materially affect the results.